# SQLGAN Branch B — Y1/Y2 lõi + Y3/Y4 renderer, Dual T4, self-contained

Notebook này clone GitHub, rồi nếu repo chưa có `sqlgan_dual` thì tự giải nén toolkit đã nhúng.

Hướng B: train Y1/Y2 core song song trên T4 x2, sau đó render Y3/Y4 từ payload core accepted.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/MinhBe/GAN_SQLi.git"
REPO_BRANCH = "main"  # repo hiện tại chủ yếu là dataset; notebook có fallback tự giải nén toolkit nếu thiếu package
REPO_DIR = Path("/kaggle/working/GAN_SQLi")

USE_GITHUB_TOKEN = True
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
PUSH_SUMMARY_TO_GIT = False

DATASET_ZIP = ""
RUN_ROOT = Path("/kaggle/working/sqlgan_runs")
SMOKE_TEST = False

MLE_EPOCHS = 12
D_EPOCHS = 4
ADV_EPOCHS = 4
ADV_STEPS = 100
GENERATE_N = 5000
BATCH_SIZE = 256
MAX_LEN = 192
EMB_DIM = 96
HIDDEN_DIM = 192
NUM_WORKERS_PER_JOB = 2
TEMPERATURE = 0.90
STATIC_WEIGHT = 0.35
COMPILE_MODEL = False

PER_PARENT = 2


In [ ]:
import os, sys, subprocess, platform, json, time, shutil, zipfile, base64
from pathlib import Path

try:
    import torch
except Exception as e:
    torch = None
    print("Không import được torch:", repr(e))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
if torch is not None:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
RUN_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
EMBEDDED_TOOLKIT_ZIP_B64 = """..."""
print("Embedded toolkit bytes:", len(EMBEDDED_TOOLKIT_ZIP_B64))


In [ ]:
import os, sys, subprocess, shutil, zipfile, base64
from pathlib import Path


def get_kaggle_secret(name: str):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(str(x) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=env, check=check)


def find_project_dir(repo_dir: Path) -> Path | None:
    candidates = [
        repo_dir,
        repo_dir / "sqlgan_dual_experiment",
        repo_dir / "sqlgan_dual_experiment_optimized",
        repo_dir / "Script" / "sqlgan_dual_experiment",
    ]
    for c in candidates:
        if (c / "sqlgan_dual" / "__init__.py").exists():
            return c
    if repo_dir.exists():
        for p in repo_dir.rglob("sqlgan_dual/__init__.py"):
            return p.parent.parent
    return None


def bootstrap_embedded_toolkit() -> Path:
    zip_out = Path("/kaggle/working/sqlgan_dual_experiment_optimized_toolkit.zip")
    extract_root = Path("/kaggle/working/embedded_sqlgan_toolkit")
    if not (extract_root / "sqlgan_dual_experiment_optimized" / "sqlgan_dual" / "__init__.py").exists():
        zip_out.write_bytes(base64.b64decode(EMBEDDED_TOOLKIT_ZIP_B64.encode("ascii")))
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_out) as zf:
            zf.extractall(extract_root)
    project = extract_root / "sqlgan_dual_experiment_optimized"
    if not (project / "sqlgan_dual" / "__init__.py").exists():
        raise FileNotFoundError("Embedded toolkit giải nén xong nhưng không thấy sqlgan_dual.")
    return project

clone_url = REPO_URL
if USE_GITHUB_TOKEN:
    token = get_kaggle_secret(GITHUB_TOKEN_SECRET)
    if token and REPO_URL.startswith("https://github.com/"):
        clone_url = REPO_URL.replace("https://", f"https://{token}@")

if REPO_DIR.exists():
    run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=False)
    run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=False)
    run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=False)
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, REPO_DIR], check=False)

print("Repo HEAD:")
run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], check=False)

PROJECT_DIR = find_project_dir(REPO_DIR)
if PROJECT_DIR is None:
    print("Repo hiện tại không có package sqlgan_dual. Dùng embedded toolkit trong notebook để không lỗi FileNotFoundError.")
    PROJECT_DIR = bootstrap_embedded_toolkit()

print("PROJECT_DIR =", PROJECT_DIR)
req = PROJECT_DIR / "requirements.txt"
if req.exists():
    run([sys.executable, "-m", "pip", "install", "-q", "-r", req])
run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT_DIR])

# Kaggle đôi lúc cài editable xong nhưng kernel chưa nhìn thấy package ngay.
# Ép thêm PROJECT_DIR vào sys.path để import ổn định, kể cả khi repo chỉ chứa dataset.
PROJECT_DIR = Path(PROJECT_DIR).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.environ["PYTHONPATH"] = str(PROJECT_DIR) + (os.pathsep + os.environ.get("PYTHONPATH", "") if os.environ.get("PYTHONPATH") else "")

pkg_init = PROJECT_DIR / "sqlgan_dual" / "__init__.py"
print("Package check:", pkg_init, "exists=", pkg_init.exists())
if not pkg_init.exists():
    print("PROJECT_DIR files:", sorted([x.name for x in PROJECT_DIR.iterdir()])[:30])
    raise FileNotFoundError(f"Không thấy package sqlgan_dual tại {pkg_init}")

import importlib
sqlgan_dual = importlib.import_module("sqlgan_dual")
print("sqlgan_dual imported from:", sqlgan_dual.__file__)


In [ ]:
from pathlib import Path


def _looks_like_v41_dataset_dir(p: Path) -> bool:
    p = Path(p)
    return (p / "Dataset" / "A_generator_attack_corpus").exists() or (p / "A_generator_attack_corpus").exists()


def _candidate_dataset_dirs(root: Path):
    root = Path(root)
    if not root.exists():
        return []
    out = []
    if _looks_like_v41_dataset_dir(root):
        out.append(root)
    # Layout 1: some_parent/Dataset/A_generator_attack_corpus
    for ds in root.rglob("Dataset"):
        if ds.is_dir() and (ds / "A_generator_attack_corpus").exists():
            out.append(ds.parent)
    # Layout 2: repo root directly has A_generator_attack_corpus
    for gen in root.rglob("A_generator_attack_corpus"):
        if gen.is_dir():
            out.append(gen.parent)
    return out


def auto_find_dataset() -> Path:
    """Find dataset for training.

    Priority:
    1. DATASET_ZIP/DATASET_PATH manually set by user.
    2. Kaggle Input zip.
    3. Kaggle Input extracted folder.
    4. Cloned GitHub repo if it already contains the dataset directories.

    This avoids failing when the repo is used as the dataset source and no Kaggle
    input zip was added.
    """
    manual = str(DATASET_ZIP).strip() if DATASET_ZIP else ""
    if manual:
        p = Path(manual)
        if not p.exists():
            raise FileNotFoundError(f"DATASET_ZIP/DATASET_PATH không tồn tại: {p}")
        print("Dùng dataset thủ công:", p)
        return p

    zip_patterns = [
        "*V4_1*StaticTrainingCorpus*.zip",
        "*V4.1*StaticTrainingCorpus*.zip",
        "*SQLGAN*V4*Static*.zip",
        "*SQLGAN*Boolean*Corpus*.zip",
        "*Attack*Corpus*.zip",
    ]

    # Prefer zip in Kaggle input / working when available.
    zip_hits = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not root.exists():
            continue
        for pat in zip_patterns:
            zip_hits.extend(root.rglob(pat))
    zip_hits = sorted(set(zip_hits), key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)
    if zip_hits:
        print("Tìm thấy dataset zip:", zip_hits[0])
        return zip_hits[0]

    # Then accept extracted folders, including the cloned GitHub repository.
    dir_roots = [Path("/kaggle/input"), Path("/kaggle/working"), REPO_DIR]
    dir_hits = []
    for root in dir_roots:
        dir_hits.extend(_candidate_dataset_dirs(root))
    # de-duplicate while preserving shorter/more direct paths first
    seen = set()
    uniq = []
    for p in sorted(dir_hits, key=lambda x: (len(str(x)), str(x))):
        rp = str(p.resolve()) if p.exists() else str(p)
        if rp not in seen:
            seen.add(rp)
            uniq.append(p)
    if uniq:
        print("Không thấy zip; dùng dataset folder:", uniq[0])
        return uniq[0]

    raise FileNotFoundError(
        "Không tìm thấy dataset V4.1. Cách sửa nhanh: Add Input dataset zip trong Kaggle, "
        "hoặc để repo clone chứa A_generator_attack_corpus/, hoặc set DATASET_ZIP/DATASET_PATH thủ công."
    )


DATA_PATH = auto_find_dataset()
print("DATA_PATH =", DATA_PATH)
if DATA_PATH.is_file():
    print("size MB =", round(DATA_PATH.stat().st_size / 1024 / 1024, 2))
else:
    print("dataset folder detected; entries =", [p.name for p in DATA_PATH.iterdir()][:20])


In [ ]:
import os, sys, subprocess, time, json, shutil, zipfile
from pathlib import Path


def available_gpus() -> int:
    try:
        import torch
        return torch.cuda.device_count() if torch.cuda.is_available() else 0
    except Exception:
        return 0


def launch_module(module_id: str, gpu_id: int | None, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    # Critical for Kaggle subprocess jobs: the notebook kernel may import sqlgan_dual
    # through sys.path, but child processes do not inherit Python sys.path.
    # Export PROJECT_DIR through PYTHONPATH so `python -m sqlgan_dual.train_module`
    # works for parallel Y1/Y2 jobs.
    _old_py_path = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(PROJECT_DIR) + (os.pathsep + _old_py_path if _old_py_path else "")
    if gpu_id is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        device = "cuda:0"
    else:
        device = "cpu"
    cmd = [
        sys.executable, "-u", "-m", "sqlgan_dual.train_module",
        "--data", str(DATA_PATH), "--out", str(out_dir), "--module-id", module_id,
        "--mle-epochs", str(MLE_EPOCHS), "--d-epochs", str(D_EPOCHS), "--adv-epochs", str(ADV_EPOCHS),
        "--adv-steps", str(ADV_STEPS), "--generate-n", str(GENERATE_N), "--batch-size", str(BATCH_SIZE),
        "--max-len", str(MAX_LEN), "--emb-dim", str(EMB_DIM), "--hidden-dim", str(HIDDEN_DIM),
        "--num-workers", str(NUM_WORKERS_PER_JOB), "--temperature", str(TEMPERATURE),
        "--static-weight", str(STATIC_WEIGHT), "--device", device,
    ]
    if SMOKE_TEST:
        cmd.append("--smoke")
    if COMPILE_MODEL:
        cmd.append("--compile")
    log_path = out_dir / "kaggle_train.log"
    fh = open(log_path, "w", encoding="utf-8")
    print(f"LAUNCH {module_id} on GPU {gpu_id}: {out_dir}")
    print("PYTHONPATH head:", env["PYTHONPATH"].split(os.pathsep)[0])
    print(" ".join(cmd))
    p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
    return {"module_id": module_id, "gpu_id": gpu_id, "process": p, "log_path": log_path, "out_dir": out_dir}


def wait_jobs(jobs):
    failed = []
    while jobs:
        still = []
        for j in jobs:
            rc = j["process"].poll()
            if rc is None:
                still.append(j)
            else:
                print(f"DONE {j['module_id']} rc={rc} log={j['log_path']}")
                try:
                    lines = Path(j["log_path"]).read_text(encoding="utf-8", errors="ignore").splitlines()
                    print("\n".join(lines[-20:]))
                except Exception as e:
                    print("Không đọc được log:", e)
                if rc != 0:
                    failed.append(j)
        jobs = still
        if jobs:
            time.sleep(15)
    if failed:
        raise RuntimeError("Có job lỗi: " + ", ".join(j["module_id"] for j in failed))


def zip_dir(src: Path, dst: Path):
    if dst.exists():
        dst.unlink()
    with zipfile.ZipFile(dst, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in src.rglob("*"):
            if p.is_file():
                if p.suffix in {".pt", ".pth"}:
                    continue
                zf.write(p, p.relative_to(src.parent))
    return dst


In [ ]:
from pathlib import Path
import sys, subprocess, json, pandas as pd
RUN_DIR = RUN_ROOT / "branch_b_core_renderer_dual_t4"
RUN_DIR.mkdir(parents=True, exist_ok=True)

gpu_count = available_gpus()
print("available_gpus =", gpu_count)
core_modules = ["Y1_basic_boolean", "Y2_boolean_variation"]
if gpu_count >= 2:
    jobs = [launch_module(core_modules[0], 0, RUN_DIR / core_modules[0]), launch_module(core_modules[1], 1, RUN_DIR / core_modules[1])]
    wait_jobs(jobs)
elif gpu_count == 1:
    for m in core_modules:
        wait_jobs([launch_module(m, 0, RUN_DIR / m)])
else:
    for m in core_modules:
        wait_jobs([launch_module(m, None, RUN_DIR / m)])
print("Core train done.")


In [ ]:
core_inputs = [RUN_DIR / "Y1_basic_boolean" / "generated_static_scored.csv", RUN_DIR / "Y2_boolean_variation" / "generated_static_scored.csv"]
cmd = [sys.executable, "-u", "-m", "sqlgan_dual.render_from_core", "--core-input", str(core_inputs[0]), "--core-input", str(core_inputs[1]), "--out", str(RUN_DIR / "renderers"), "--per-parent", str(PER_PARENT)]
if SMOKE_TEST:
    cmd += ["--limit", "256"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
subprocess.run([sys.executable, "-m", "sqlgan_dual.aggregate_runs", "--run-dir", str(RUN_DIR), "--out-prefix", "branch_b_summary"], check=True)
print("RUN_DIR =", RUN_DIR)


In [ ]:
ZIP_PATH = Path("/kaggle/working/branch_b_core_renderer_summary.zip")
zip_dir(RUN_DIR, ZIP_PATH)
print("ZIP_PATH =", ZIP_PATH)
for p in sorted(RUN_DIR.rglob("*summary*.csv")):
    print(p)


In [ ]:
if PUSH_SUMMARY_TO_GIT:
    import subprocess, os, shutil
    summary_dir = RUN_DIR / "git_summary_export"
    if summary_dir.exists(): shutil.rmtree(summary_dir)
    summary_dir.mkdir(parents=True, exist_ok=True)
    for p in list(RUN_DIR.rglob("*.csv")) + [x for x in RUN_DIR.rglob("*.json") if "checkpoint" not in x.parts]:
        rel = p.relative_to(RUN_DIR)
        dst = summary_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_bytes(p.read_bytes())
    target = REPO_DIR / "kaggle_runs" / RUN_DIR.name
    if target.exists(): shutil.rmtree(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(summary_dir, target)
    subprocess.run(["git", "-C", REPO_DIR, "add", str(target.relative_to(REPO_DIR))], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "commit", "-m", f"Add Kaggle summary {RUN_DIR.name}"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", REPO_BRANCH], check=False)


Branch B dùng GPU chủ yếu ở Y1/Y2 core. Y3/Y4 là lớp biến đổi có kiểm soát, nên không cần train GAN riêng.
